# French Baby Names

First names registered in France by department, 1900-2020
([dpt2020.csv](https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv)).

1. Evolution over time - line chart with search
2. Regional spread - choropleth map
3. Gender split - centered butterfly chart

In [1]:
# !pip install -r requirements.txt

In [2]:
import json
import os
import time
from urllib.request import urlopen

import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# Each chart is exported as a standalone HTML file in this folder
os.makedirs('exports', exist_ok=True)

## Data

Load the CSV, then keep valid years and real names.

In [3]:
url = "https://perso.telecom-paristech.fr/eagan/class/igr204/data/dpt2020.csv"

# ~79 MB; the server connection sometimes drops mid-download, so fetch it once to a
# local cache (with a few retries) and read from there. Delete dpt2020.csv to refresh.
csv_file = "dpt2020.csv"
if not os.path.exists(csv_file):
    for attempt in range(5):
        try:
            with urlopen(url) as resp, open(csv_file, "wb") as f:
                f.write(resp.read())
            break
        except Exception:
            if os.path.exists(csv_file):
                os.remove(csv_file)
            if attempt == 4:
                raise
            time.sleep(3)

df = pd.read_csv(csv_file, sep=';', dtype={'annais': str, 'dpt': str})
df.head()

,sexe,preusuel,annais,dpt,nombre
0,1,_PRENOMS_RARES,1900,02,7
1,1,_PRENOMS_RARES,1900,04,9
2,1,_PRENOMS_RARES,1900,05,8
3,1,_PRENOMS_RARES,1900,06,23
4,1,_PRENOMS_RARES,1900,07,9


In [4]:
df.columns = ['sex', 'name', 'year', 'dept', 'births']
df = df[df.name != '_PRENOMS_RARES']
df = df[df.year != 'XXXX']
df['year'] = df.year.astype(int)

## 1. Evolution over time

The 300 most common names are shown in grey. Type names separated by `|` to highlight them.

In [5]:
yearly = df.groupby(['name', 'year'], as_index=False).births.sum()

top = yearly.groupby('name').births.sum().nlargest(300).index
pool = yearly[yearly.name.isin(top)]

In [6]:
box = alt.binding(input='text', name='Highlight: ')
search = alt.param(name='q', value='JEAN|MARIE|KEVIN', bind=box)
match = "test(regexp('^(' + q + ')$', 'i'), datum.name)"

base = alt.Chart(pool).encode(
    x='year:Q',
    y=alt.Y('births:Q', scale=alt.Scale(type='log')),
    detail='name:N')

grey = base.mark_line(color='lightgray')
bold = base.mark_line().encode(color='name:N').transform_filter(match)

viz1 = (grey + bold).add_params(search).properties(width=700, height=400)
viz1.save('exports/Viz1.html', inline=True)
print('Exported to exports/Viz1.html')

Exported to exports/Viz1.html


## 2. Regional spread

- Pick a name and a year. Each department is coloured by the **number of babies given that name there
that year** -- the darker, the more.
- The scale is fixed at **0 to 1000+** (1000 or more shows the
darkest shade), the same for every name and year. Exact count in the tooltip.

### Viz 2:  changes made, per peer review from the forum

| Reviewer | Comment on the forum | Change made |
|---|---|---|
| Ambroise & Yassine | the colour scale moves across years | colour scale **fixed at 0 to 1000+**, identical for every name and year |
| Andre & Anne | clunky to compare two names | added a **side-by-side** comparison of two names, on a **shared year** |
| Anne | sort the name dropdown alphabetically | dropdown **sorted alphabetically** |
| Nathan | use percentages instead of raw values | we **kept raw counts** (most direct reading) — *not adopted* |

In [7]:
# --- Viz 2 data: raw births per (name, dept, year) for a curated set of names ---
geo_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"

# small file, but use the same robust fetch-to-cache pattern as the CSV
geo_file = "france.json"
if not os.path.exists(geo_file):
    for attempt in range(5):
        try:
            with urlopen(geo_url) as resp, open(geo_file, "wb") as f:
                f.write(resp.read())
            break
        except Exception:
            if os.path.exists(geo_file):
                os.remove(geo_file)
            if attempt == 4:
                raise
            time.sleep(2)
with open(geo_file, encoding='utf-8') as r:
    france = json.load(r)

by_dept = df.groupby(['name', 'dept', 'year'], as_index=False).births.sum()

# Corsica is '20' in the data but 2A / 2B on the map
corse = by_dept[by_dept.dept == '20']
for code in '2A', '2B':
    by_dept = pd.concat([by_dept, corse.assign(dept=code)])

# curated names (Breton, Basque, national, trend-driven), sorted for the dropdown
picks = sorted(['KEVIN', 'ERWAN', 'MAEL', 'RONAN', 'AITOR', 'MAITE', 'MARIE', 'JEAN', 'MOHAMED'])
by_dept = by_dept[by_dept.name.isin(picks)][['name', 'dept', 'year', 'births']]

In [8]:
# --- Viz 2 maps: colour = raw births of a name in a department, fixed 0-1000+ scale ---
geo = alt.InlineData(values=france['features'])
lookup = alt.LookupData(geo, 'properties.code', ['type', 'geometry', 'properties'])
bg = alt.Chart(geo).mark_geoshape(fill='lightgray', stroke='white')

def carte(name_param, year_param, size, title):
    """One choropleth, driven by the params named `name_param` and `year_param`."""
    return (bg + alt.Chart(by_dept)
            .transform_filter(f'datum.name == {name_param} && datum.year == {year_param}')
            .transform_lookup('dept', from_=lookup).mark_geoshape(stroke='white')
            .encode(
                color=alt.Color('births:Q',
                                scale=alt.Scale(scheme='blues', domain=[0, 1000], clamp=True),
                                legend=alt.Legend(title='Babies named this',
                                                  labelExpr="datum.value >= 1000 ? '1000+' : datum.label")),
                tooltip=[alt.Tooltip('properties.nom:N', title='Department'),
                         alt.Tooltip('births:Q', title='Babies named this', format=',')])
            ).project('mercator').properties(width=size, height=size, title=title)

# one map: pick a name + year
name_sel = alt.param(name='sel_name', value='KEVIN', bind=alt.binding_select(options=picks, name='Name: '))
year_sel = alt.param(name='sel_year', value=1991, bind=alt.binding_range(min=1900, max=2020, step=1, name='Year: '))
(carte('sel_name', 'sel_year', 520, 'Regional spread')
 .add_params(name_sel, year_sel).save('exports/Viz2.html', inline=True))

# side-by-side: two names, one shared year
nameL = alt.param(name='nameL', value='KEVIN', bind=alt.binding_select(options=picks, name='Left name: '))
nameR = alt.param(name='nameR', value='JEAN', bind=alt.binding_select(options=picks, name='Right name: '))
cmp_year = alt.param(name='cmp_year', value=1991, bind=alt.binding_range(min=1900, max=2020, step=1, name='Year (shared): '))
((carte('nameL', 'cmp_year', 330, 'Left') | carte('nameR', 'cmp_year', 330, 'Right'))
 .resolve_scale(color='shared').add_params(nameL, nameR, cmp_year)
 .save('exports/Viz2_comparison.html', inline=True))

print('Exported exports/Viz2.html and exports/Viz2_comparison.html')

Exported exports/Viz2.html and exports/Viz2_comparison.html


## 3. Gender split


### Viz 3: Centered Butterfly Chart (Improvements)

Based on the peer feedback received, the following improvements were made to this visualization:

* **Enforced Symmetrical Scaling:** Addressed feedback regarding distorted axes by using an "invisible bounds" trick (`transform_joinaggregate` and transparent marks). This calculates the maximum absolute births for the selected name and forces the X-axis to scale symmetrically, ensuring the zero-line stays perfectly centered.
* **Dynamic Unisex Pre-selection Filter:** Replaced the static, hardcoded list of names with a dynamic Pandas calculation. The dropdown now automatically filters for and displays only truly unisex names (where the minority gender accounts for >10% of total births, with a baseline of >20,000 total births).
* **Enhanced Tooltips with Percentage Share:** Clarified the evolution of the gender split by adding calculations to the tooltip that compute and display the exact percentage share (e.g., "87.0%") alongside the absolute volume of births.
* **Improved Temporal Granularity:** Refined the Y-axis grouping from 20-year blocks to standard 10-year decades to provide better visual granularity for tracking rapid historical shifts.

In [9]:
gender = df.groupby(['name', 'sex', 'year'], as_index=False).births.sum()
gender['decade'] = (gender.year // 10) * 10
gender = gender.groupby(['name', 'sex', 'decade'], as_index=False).births.sum()

# --- Dynamic unisex name detection ---
# Total births per name per sex, pivoted so each sex is a column
sex_totals = (
    df.groupby(['name', 'sex']).births.sum()
    .unstack(fill_value=0)
    .rename(columns={1: 'male', 2: 'female'})
)
sex_totals['total'] = sex_totals['male'] + sex_totals['female']
sex_totals['minority_pct'] = sex_totals[['male', 'female']].min(axis=1) / sex_totals['total']

bfly_names = (
    sex_totals[
        (sex_totals['minority_pct'] > 0.10) &
        (sex_totals['total'] > 20_000)
    ]
    .index.sort_values()
    .tolist()
)

gender = gender[gender.name.isin(bfly_names)]

name_box = alt.binding_select(options=bfly_names, name='Name: ')
name_pick = alt.param(name='bf_name', value=bfly_names[0], bind=name_box)

base = (
    alt.Chart(gender)
    .transform_filter('datum.name == bf_name')
    .transform_joinaggregate(max_b='max(births)')
    .transform_joinaggregate(decade_total='sum(births)', groupby=['decade'])
    .transform_calculate(
        val='datum.sex == 1 ? -datum.births : datum.births',
        label='datum.sex == 1 ? "Male" : "Female"',
        sym_max='datum.max_b',
        sym_min='-datum.max_b',
        pct='datum.births / datum.decade_total'
    )
)

bars = base.mark_bar().encode(
    y=alt.Y('decade:O', sort='descending', title=None),
    x=alt.X('val:Q', title='Births', axis=alt.Axis(labelExpr='abs(datum.value)')),
    color=alt.Color('label:N',
        scale=alt.Scale(domain=['Male', 'Female'], range=['steelblue', 'darkorange']),
        legend=alt.Legend(orient='top', title=None)),
    tooltip=[alt.Tooltip('label:N', title='Sex'),
             alt.Tooltip('decade:O', title='Decade'),
             alt.Tooltip('births:Q', title='Births'),
             alt.Tooltip('pct:Q', title='Share', format='.1%')]
)

dummy_min = base.mark_point(opacity=0).encode(
    y=alt.Y('decade:O', sort='descending'),
    x=alt.X('sym_min:Q')
)

dummy_max = base.mark_point(opacity=0).encode(
    y=alt.Y('decade:O', sort='descending'),
    x=alt.X('sym_max:Q')
)

viz3 = (
    alt.layer(bars, dummy_min, dummy_max)
    .add_params(name_pick)
    .properties(width=500, height=300, title='Gender split over time')
)
viz3.save('exports/Viz3.html', inline=True)
print('Exported to exports/Viz3.html')



Exported to exports/Viz3.html
